In [ ]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 2000.0
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- 1. Physical Parameters ---
N = 512
beta_val = 1.0

dt = 0.01  # Time step for numerical stability
total_steps_elapsed = 0
n_frames = 250
steps_per_frame = 100

# Harmonic driving parameters (in the forbidden band gap)
omega_driver = 2.12
initial_displacement = 0.287  # Driving amplitude (a)

# --- 2. Initial Conditions & History Arrays ---
u = np.zeros(N)
v = np.zeros(N)

u_history = np.zeros((n_frames, N))
v_history = np.zeros((n_frames, N))

beta_mask = np.ones(N - 1) * beta_val


def get_acceleration(pos):
    """Compute accelerations for the 1D nonlinear beta-FPU chain: F = du + beta * du^3."""
    acc = np.zeros(N)
    diff = np.diff(pos)
    forces = diff + beta_mask * (diff ** 3)
    
    # Internal free particles (indices 2 to N-2); u[1] is externally driven
    acc[2:-1] = forces[2:] - forces[1:-1]
    return acc


# --- 3. Visualization Setup ---
fig, ax = plt.subplots(figsize=(12, 6))
indices = np.arange(N)

line, = ax.plot(indices[1:400], u[1:400], 'o-', lw=1, markersize=2, color='#2980b9', alpha=0.5)
driven_bead, = ax.plot(indices[1], u[1], 'o', markersize=9, color='#e74c3c', label='Driven Bead (u1)')
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)

ax.set_ylim(-0.6, 0.6)
ax.set_xlim(0, 400)
ax.set_xlabel("Lattice Site (n)")
ax.set_ylabel("Displacement (u)")
ax.set_title("Fully Nonlinear beta-FPU Chain: Driven Continuous Soliton Generation")
ax.legend(loc='upper right')
ax.grid(True, alpha=0.2)


def update(frame):
    """Advance the system using the symplectic Velocity Verlet integrator."""
    global u, v, total_steps_elapsed

    for _ in range(steps_per_frame):
        t = total_steps_elapsed * dt

        # 1. Half-step velocity update for free particles
        a = get_acceleration(u)
        v[2:-1] += 0.5 * a[2:-1] * dt

        # 2. Full-step position update for free particles
        u[2:-1] += v[2:-1] * dt

        # 3. Harmonic boundary driving at site n = 1
        t_next = t + dt
        u[1] = initial_displacement * np.sin(omega_driver * t_next)
        v[1] = initial_displacement * omega_driver * np.cos(omega_driver * t_next)

        # Fixed boundary walls
        u[0], v[0] = 0.0, 0.0
        u[-1], v[-1] = 0.0, 0.0

        # 4. Second half-step velocity update
        a_new = get_acceleration(u)
        v[2:-1] += 0.5 * a_new[2:-1] * dt

        total_steps_elapsed += 1

    # Record state history
    u_history[frame, :] = u.copy()
    v_history[frame, :] = v.copy()

    # Update animation artists
    line.set_ydata(u[1:400])
    driven_bead.set_ydata([u[1]])
    time_text.set_text(f"Time: {total_steps_elapsed * dt:.2f}s")

    if frame % 50 == 0:
        print(f"Frame: {frame} / Time: {total_steps_elapsed * dt:.1f}s")

    return [line, driven_bead, time_text]


from matplotlib.animation import PillowWriter

ani = FuncAnimation(fig, update, frames=n_frames, interval=30, blit=False)
plt.close(fig)

# dpi=70 და fps=15 ზომას 3-5 MB-მდე ჩამოიყვანს, რასაც GitHub უპრობლემოდ აჩვენებს
ani.save("soliton_animation.gif", writer=PillowWriter(fps=15), dpi=70)

In [ ]:
from scipy.fft import fft, fftfreq


def analyze_temporal_fft(u_hist, bead_idx, dt_sample=1.0):
    """Compute and plot the time-domain trajectory and temporal FFT for a given lattice site."""
    x = u_hist[:, bead_idx]
    n_points = len(x)
    t = np.arange(1, n_points + 1) * dt_sample

    # Compute FFT and positive angular frequencies (omega = 2 * pi * f)
    X_complex = fft(x)
    frequencies = fftfreq(n_points, dt_sample)
    half_N = n_points // 2

    omega_positive = 2 * np.pi * frequencies[:half_N]
    amplitude_spectrum = (2.0 / n_points) * np.abs(X_complex[:half_N])

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(t, x, color="blue")
    axes[0].set_title(f"Time-Domain Signal u_{bead_idx}(t)")
    axes[0].set_xlabel("Time (t)")
    axes[0].set_ylabel("Displacement")
    axes[0].grid(True, alpha=0.3)

    axes[1].stem(omega_positive, amplitude_spectrum, basefmt=" ", markerfmt="ro")
    axes[1].set_title(f"Temporal Amplitude Spectrum (Site n = {bead_idx})")
    axes[1].set_xlabel("Angular Frequency (rad/s)")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_xlim(0, 3)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    max_amp = np.max(amplitude_spectrum)
    dom_omega = omega_positive[np.argmax(amplitude_spectrum)]
    print(f"Site n = {bead_idx} | Max amplitude: {max_amp:.5f} | Dominant frequency: {dom_omega:.5f} rad/s")
    return dom_omega, omega_positive, amplitude_spectrum


# Analyze the driven bead (n = 1) and an internal bead (n = 65)
analyze_temporal_fft(u_history, bead_idx=1, dt_sample=1.0)
analyze_temporal_fft(u_history, bead_idx=65, dt_sample=1.0)

In [ ]:
from scipy.fft import rfft, rfftfreq


def analyze_spatial_fft_window(u_hist, bead_start, bead_end, frame_start, frame_end):
    """
    Compute the spatial FFT over lattice sites [bead_start, bead_end]
    averaged across frames [frame_start, frame_end].
    """
    window_data = u_hist[frame_start:frame_end, bead_start:bead_end]
    n_beads = window_data.shape[1]

    if n_beads < 4:
        raise ValueError("Selected spatial window is too small for FFT analysis.")

    dx = 1.0  # Lattice spacing
    k_spatial = 2 * np.pi * rfftfreq(n_beads, d=dx)

    # Apply Hanning window to suppress spectral leakage
    window_func = np.hanning(n_beads)
    windowed_signals = window_data * window_func

    fft_matrix = rfft(windowed_signals, axis=1)
    amp_matrix = (2.0 / n_beads) * np.abs(fft_matrix)
    amp_matrix[:, 0] = 0.0  # Exclude DC component (p = 0)

    mean_spectrum = np.mean(amp_matrix, axis=0)
    dominant_idx = np.argmax(mean_spectrum)
    p_found = k_spatial[dominant_idx]

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    im = axes[0].imshow(
        window_data, aspect='auto', cmap='coolwarm', origin='lower',
        extent=[bead_start, bead_end, frame_start, frame_end]
    )
    axes[0].set_title(f"Space-Time Window: Sites {bead_start}-{bead_end}")
    axes[0].set_xlabel("Lattice Site (n)")
    axes[0].set_ylabel("Frame Index")
    fig.colorbar(im, ax=axes[0], label="Displacement (u)")

    axes[1].plot(k_spatial, mean_spectrum, 'r.-', lw=1.5, markersize=5)
    axes[1].axvline(p_found, color='black', linestyle='--', label=f"Dominant p = {p_found:.4f} rad")
    axes[1].set_title(f"Mean Spatial Spectrum (Frames {frame_start}-{frame_end})")
    axes[1].set_xlabel("Wavenumber p (rad)")
    axes[1].set_ylabel("Amplitude")
    axes[1].set_xlim(0, np.pi)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("--- Spatial FFT Results ---")
    print(f"Window: Sites [{bead_start}:{bead_end}], Frames [{frame_start}:{frame_end}]")
    print(f"Dominant wavenumber p: {p_found:.5f} rad")
    print(f"Wavelength lambda: {2 * np.pi / p_found:.2f} lattice sites")

    return p_found, k_spatial, mean_spectrum


p_sim, k_axis, spectrum = analyze_spatial_fft_window(
    u_hist=u_history,
    bead_start=50,
    bead_end=75,
    frame_start=190,
    frame_end=200
)

In [ ]:
def calculate_flux_at_n(u_history, v_history, n, beta=1.0):
    """
    Compute the local energy flux j_n(t) at lattice site n over time.

    Parameters:
        u_history: 2D array of displacements (time, sites)
        v_history: 2D array of velocities (time, sites)
        n: Lattice site index
        beta: Nonlinearity coefficient
    """
    u_hist = np.asarray(u_history)
    v_hist = np.asarray(v_history)

    avg_velocity = 0.5 * (v_hist[:, n] + v_hist[:, n + 1])
    du = u_hist[:, n + 1] - u_hist[:, n]

    j_n = avg_velocity * (du + beta * (du ** 3))
    return j_n


t_frames = np.arange(1, len(u_history) + 1, dtype=float)
j_data = calculate_flux_at_n(u_history, v_history, n=64, beta=beta_val)
E_soliton = np.trapezoid(j_data, t_frames)

print(f"Total transmitted soliton energy at site n = 64: {E_soliton:.6f}")